In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from imblearn.under_sampling import RandomUnderSampler
from imblearn.under_sampling import TomekLinks
from imblearn.under_sampling import EditedNearestNeighbours
from imblearn.over_sampling import RandomOverSampler
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import SMOTENC
from imblearn.over_sampling import KMeansSMOTE
from imblearn.combine import SMOTEENN
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline
import random
from sklearn_extra.cluster import KMedoids
import warnings
import os
warnings.filterwarnings("ignore")

Functions for resampling

In [2]:
np.random.seed(42)
def K_Medoids(X,n_clusters,option='maj'):
    kmed=KMedoids(n_clusters=n_clusters).fit(X)
    X_temp=X.copy()
    if option=='maj':
        labels=pd.Series(kmed.labels_).value_counts().to_dict()
        maj_value=max(labels.values())
        maj_label=max(labels,key=labels.get)
        X_biggest_class=X[kmed.labels_==maj_label]

        index=random.sample(X_biggest_class.index.values.tolist(),1)[0]
        X_temp.drop(index,axis=0,inplace=True,errors='IgnoreRaise')
        
    elif option=='min':
        labels=pd.Series(kmed.labels_).value_counts().to_dict()
        min_value=min(labels.values())
        min_label=min(labels,key=labels.get)
        X_smallest_class=X[kmed.labels_==min_label]

        index=random.sample(X_smallest_class.index.values.tolist(),1)[0]

        names=X_smallest_class.index.tolist().index(index)
       
        X_temp=X_temp._append(X_temp.iloc[names])
    return X_temp
        
    
def K_Medoids_RUS(X,y,ratio,n_clusters):
    all_class=class_majority_dict_for_sampling(X,y,1)
    classes=list(pd.Series(y).value_counts().to_dict().keys())
    
    class_dict=class_majority_dict_for_sampling(X,y,ratio)
    class_maj=[k for k in class_dict.keys()][0]
    class_min=[k for k in classes if k!=class_maj][0]
    
    X_min=X[y==class_min]
    X_maj=X[y==class_maj]
    num_to_undersample= all_class[class_maj]- class_dict[class_maj]

    for k in range(num_to_undersample):
        X_maj=K_Medoids(X_maj,n_clusters,option='maj')

    final_df=pd.concat([X_maj,X_min],axis=0)
    return [final_df,y[final_df.index]]

def K_Medoids_ROS(X,y,ratio,n_clusters):
    minn=pd.Series(y).value_counts().to_dict()
    min_label=min(minn,key=minn.get)
    y_min=minn[min_label]
    classes=list(pd.Series(y).value_counts().to_dict().keys())

   
    
    class_dict=class_minority_dict_for_sampling(X,y,ratio)
    
    class_min=[k for k in class_dict.keys()][0]
    class_maj=[k for k in classes if k!=class_min][0]

    
    X_min=X[y==class_min]
    X_maj=X[y==class_maj]
    
    num_to_oversample= class_dict[class_min] - y_min
    
    for k in range(num_to_oversample):
        X_min1=X_min.copy()
        X_min=K_Medoids(X_min1,n_clusters,option='min')
    #return X_min
    final_df=pd.concat([X_maj,X_min],axis=0)
    return [final_df,y[final_df.index]]
    
def class_minority_dict_for_sampling(X,y,ratio):
    ### sampling strategy function for over-sampling
    classes=pd.Series(y).value_counts().to_dict()
    maj_class=max(classes.values())
    ####Number of desired samples for minority classes
    min_sampls = maj_class*ratio
    maj_key=max(classes,key=classes.get)
    min_class_dict_before = {key: value for key, value in classes.items() if key != maj_key}
    if min_sampls < max(min_class_dict_before.values()):
        min_sampls=int(max(min_class_dict_before.values()))
    for key in classes:
        if key != maj_key:
            classes[key]=int(min_sampls)
    class_dict = {key: int(value) for key, value in classes.items() if key != maj_key}

    return class_dict

def class_majority_dict_for_sampling(X,y,ratio):
    ### sampling strategy function for over-sampling
    classes=pd.Series(y).value_counts().to_dict()
    maj_class=max(classes.values())
    maj_key=max(classes,key=classes.get)
    maj_samples=int(maj_class)*ratio
    class_dict={maj_key :int(maj_samples)}
    
    return class_dict
###1. Undersampling
###1.1 Random Under-Sampling
def RUS(X,y,ratio=0.5):
    if ratio =='':
        rus=RandomUnderSampler(replacement=True)
        X_re,y_re=rus.fit_resample(X,y)
        return [X_re,y_re]
    classes=class_majority_dict_for_sampling(X,y,ratio)
    rus=RandomUnderSampler(sampling_strategy=classes,replacement=True)
    X_re,y_re=rus.fit_resample(X,y)
    return [X_re,y_re]
###1.2 Tomek Links
def TL(X,y):
    tl=TomekLinks(sampling_strategy='majority')
    X_re,y_re=tl.fit_resample(X,y)
    return [X_re,y_re]

###1.3 Edited Nearest Neighbors
def ENN(X,y):
    enn=EditedNearestNeighbours(sampling_strategy='majority')
    X_re,y_re=enn.fit_resample(X,y)
    return [X_re,y_re]
    
###2. Oversampling
###2.1 Random Over-Sampling
def ROS(X,y,ratio=0.5):
    classes=class_minority_dict_for_sampling(X,y,ratio)
    ros=RandomOverSampler(sampling_strategy=classes)
    X_re,y_re=ros.fit_resample(X,y)
    return [X_re,y_re]
###2.2 SMOTE
def smote_fnc(X,y,type_smote='numerical',ratio=0.5,categorical_col_inds=[]):
    classes=class_minority_dict_for_sampling(X,y,ratio)
    if type_smote=='numerical':
        smote1=SMOTE(sampling_strategy=classes,k_neighbors=7)
    else:
        smote1=SMOTENC(categorical_features=categorical_col_inds,sampling_strategy=classes,random_state=42)
    X_re,y_re=smote1.fit_resample(X,y)
    return [X_re,y_re]
###2.3 K-Means smote
def kmeansSMOTE(X,y):
    kmeans=KMeansSMOTE()
    X_re,y_re=kmeans.fit_resample(X,y)
    return [X_re,y_re]
###3. Hybrid methods
###3.1 SMOTE-ENN
def SMOTE_ENN(X,y,ratio=0.5,type_smote='numerical',categorical_feats=[]):
    min_class_dict = class_minority_dict_for_sampling(X,y,ratio)
    if type_smote=='numerical':
        enn_smote=SMOTEENN(sampling_strategy=min_class_dict)
    else:
        enn_smote=SMOTEENN(sampling_strategy=min_class_dict,smote=SMOTENC(categorical_features=categorical_feats,k_neighbors=7))
    X_re,y_re=enn_smote.fit_resample(X,y)
    return [X_re,y_re]
###3.2 SMOTE-TL
def SMOTE_TL(X,y,ratio=0.5,type_smote='numerical',categorical_feats=[]):
    min_class_dict = class_minority_dict_for_sampling(X,y,ratio)
    if type_smote=='numerical':
        smotetl=SMOTETomek(sampling_strategy=min_class_dict)
    else:
        smotetl=SMOTETomek(sampling_strategy=min_class_dict,smote=SMOTENC(categorical_features=categorical_feats,k_neighbors=7))
    smotetl=SMOTETomek(sampling_strategy=min_class_dict)
    X_re,y_re=smotetl.fit_resample(X,y)
    return [X_re,y_re]

###3.3 SMOTE-RUS
def SMOTE_RUS(X,y,ratio_under=0.5,ratio_over=0.5,type_smote='numerical',categorical_feats=[]):
    maj_class_dict=class_majority_dict_for_sampling(X,y,ratio_under)
    
    min_class_dict=class_minority_dict_for_sampling(X,y,ratio_over)
    
    if type_smote=='numerical':
        over=SMOTE(sampling_strategy=min_class_dict)
    else:
        over=SMOTENC(sampling_strategy=min_class_dict,categorical_features=categorical_feats,k_neighbros=7)

    under=RandomUnderSampler(sampling_strategy=maj_class_dict)
         
    steps=[('u',under),('o',over)]
    pipeline=Pipeline(steps=steps)
    X_re,y_re=pipeline.fit_resample(X,y)
    return [X_re,y_re]

In [ ]:
os.chdir(r'.\Train_test_sets')
train_sets=os.listdir(os.getcwd())

#train_sets
cads_files = [f for f in train_sets if "_CADS_" in f]
cadp_files = [f for f in train_sets if "_CADP_" in f]
cadhs_files = [f for f in train_sets if "_CADHS_" in f]
cads_train_files=[f for f in cads_files if "Train_" in f]
cadp_train_files=[f for f in cadp_files if "Train_" in f]
cadhs_train_files=[f for f in cadhs_files if "Train_" in f] 

In [11]:
cad_all_files=[f for f in train_sets if "Whole_" in f]

In [12]:
def categorical_indices(data):
    categorical_cols=[]
    for i,col in enumerate(data.columns):
        if data[col].dtype=='int64':
            categorical_cols.append(i)
    return categorical_cols

In [13]:
### CAD Presence : CADP
for file in cadp_train_files:
    print(file)
    data=pd.read_csv(file)
    data.drop('SYNTAX.SCORE',axis=1,inplace=True)
    train_set=data
    y_train=train_set['SYNTAX.BINARY']
    X_train=train_set.iloc[:, :-1]
    ratio_under=min(y_train.value_counts())/max(y_train.value_counts())
    categorical_inds=categorical_indices(X_train)
    [X_train_rus,y_train_rus]=RUS(X_train,y_train,ratio_under)

    ### RUS
    new_filename = file.replace('_CADP_orig', '_CADP_rus')
    X_train_rus['SYNTAX.BINARY']=y_train_rus
    X_train_rus.to_csv(new_filename,index=False)

    ### ENN
    [X_train_enn,y_train_enn]=ENN(X_train,y_train)
    new_filename = file.replace('_CADP_orig', '_CADP_enn')
    X_train_enn['SYNTAX.BINARY']=y_train_enn
    X_train_enn.to_csv(new_filename,index=False)

    ### ROS
    [X_train_ros,y_train_ros]=ROS(X_train,y_train,1)
    new_filename = file.replace('_CADP_orig', '_CADP_ros')
    X_train_ros['SYNTAX.BINARY']=y_train_ros
    X_train_ros.to_csv(new_filename,index=False)

    ### SMOTE
    [X_train_smote,y_train_smote]=smote_fnc(X_train,y_train,ratio=1,type_smote='categorical',categorical_col_inds=categorical_inds)
    new_filename = file.replace('_CADP_orig', '_CADP_smote')
    X_train_smote['SYNTAX.BINARY']=y_train_smote
    X_train_smote.to_csv(new_filename,index=False)
    ### SMOTE-ENN
    [X_train_smote_enn,y_train_smote_enn]=SMOTE_ENN(X_train,y_train,ratio=1,type_smote='categorical',categorical_feats=categorical_inds)
    new_filename = file.replace('_CADP_orig', '_CADP_smote_enn')
    X_train_smote_enn['SYNTAX.BINARY']=y_train_smote_enn
    X_train_smote_enn.to_csv(new_filename,index=False)
    ###### Need to address correctly the indexes of the categorical columns.... 

Train_set_CADP_orig_1.csv
Train_set_CADP_orig_10.csv
Train_set_CADP_orig_2.csv
Train_set_CADP_orig_3.csv
Train_set_CADP_orig_4.csv
Train_set_CADP_orig_5.csv
Train_set_CADP_orig_6.csv
Train_set_CADP_orig_7.csv
Train_set_CADP_orig_8.csv
Train_set_CADP_orig_9.csv


In [7]:
### Whole dataset experiment
for file in cad_all_files:
    print(file)
    data=pd.read_csv(file)
    data.drop('ID',axis=1,inplace=True)
    train_set=data
    if(file=='Whole_set_CADP_orig_1.csv'):

        y_train=train_set['SYNTAX.BINARY']
        X_train=train_set.iloc[:, :-4]
        ratio_under=min(y_train.value_counts())/max(y_train.value_counts())
        categorical_inds=categorical_indices(X_train)
        [X_train_rus,y_train_rus]=RUS(X_train,y_train,ratio_under)

        ### RUS
        new_filename = file.replace('_orig', '_rus')
        X_train_rus['SYNTAX.BINARY']=y_train_rus
    elif(file=='Whole_set_CADS_orig_1.csv'):
        y_train=train_set['SYNTAX.SEVERITY']
        X_train=train_set.iloc[:, :-4]
        ratio_under=min(y_train.value_counts())/max(y_train.value_counts())
        categorical_inds=categorical_indices(X_train)
        [X_train_rus,y_train_rus]=RUS(X_train,y_train,ratio_under)

        ### RUS
        new_filename = file.replace('_orig', '_rus')
        X_train_rus['SYNTAX.SEVERITY']=y_train_rus
    elif(file=='Whole_set_CADHS_orig_1.csv'):
        y_train=train_set['SYNTAX.HIGH.SEVERITY']
        X_train=train_set.iloc[:, :-4]
        ratio_under=min(y_train.value_counts())/max(y_train.value_counts())
        categorical_inds=categorical_indices(X_train)
        [X_train_rus,y_train_rus]=RUS(X_train,y_train,ratio_under)

        ### RUS
        new_filename = file.replace('_orig', '_rus')
        X_train_rus['SYNTAX.HIGH.SEVERITY']=y_train_rus

    X_train_rus.to_csv(new_filename,index=False)

    

In [14]:
### CAD Severity : CADS
for file in cads_train_files:
    print(file)
    data=pd.read_csv(file)
    data.drop('SYNTAX.SCORE',axis=1,inplace=True)
    train_set=data
    y_train=train_set['SYNTAX.SEVERITY']
    X_train=train_set.iloc[:, :-1]
    ratio_under=min(y_train.value_counts())/max(y_train.value_counts())
    categorical_inds=categorical_indices(X_train)
    [X_train_rus,y_train_rus]=RUS(X_train,y_train,ratio_under)

    ### RUS
    new_filename = file.replace('_CADS_orig', '_CADS_rus')
    X_train_rus['SYNTAX.SEVERITY']=y_train_rus
    X_train_rus.to_csv(new_filename,index=False)

    ### ENN
    [X_train_enn,y_train_enn]=ENN(X_train,y_train)
    new_filename = file.replace('_CADS_orig', '_CADS_enn')
    X_train_enn['SYNTAX.SEVERITY']=y_train_enn
    X_train_enn.to_csv(new_filename,index=False)

    ### ROS
    [X_train_ros,y_train_ros]=ROS(X_train,y_train,1)
    new_filename = file.replace('_CADS_orig', '_CADS_ros')
    X_train_ros['SYNTAX.SEVERITY']=y_train_ros
    X_train_ros.to_csv(new_filename,index=False)

    ### SMOTE
    if(len(categorical_inds)>0):
        [X_train_smote,y_train_smote]=smote_fnc(X_train,y_train,ratio=1,type_smote='categorical',categorical_col_inds=categorical_inds)
    else:
        [X_train_smote,y_train_smote]=smote_fnc(X_train,y_train,ratio=1,type_smote='numerical',categorical_col_inds=[])
    new_filename = file.replace('_CADS_orig', '_CADS_smote')
    X_train_smote['SYNTAX.SEVERITY']=y_train_smote
    X_train_smote.to_csv(new_filename,index=False)
    ### SMOTE-ENN
    if(len(categorical_inds)>0):
        [X_train_smote_enn,y_train_smote_enn]=SMOTE_ENN(X_train,y_train,ratio=1,type_smote='categorical',categorical_feats=categorical_inds)
    else:
        [X_train_smote_enn,y_train_smote_enn]=SMOTE_ENN(X_train,y_train,ratio=1,type_smote='numerical',categorical_feats=[])
    new_filename = file.replace('_CADS_orig', '_CADS_smote_enn')
    X_train_smote_enn['SYNTAX.SEVERITY']=y_train_smote_enn
    X_train_smote_enn.to_csv(new_filename,index=False)
    ###### Need to address correctly the indexes of the categorical columns.... 

Train_set_CADS_orig_1.csv
Train_set_CADS_orig_10.csv
Train_set_CADS_orig_2.csv
Train_set_CADS_orig_3.csv
Train_set_CADS_orig_4.csv
Train_set_CADS_orig_5.csv
Train_set_CADS_orig_6.csv
Train_set_CADS_orig_7.csv
Train_set_CADS_orig_8.csv
Train_set_CADS_orig_9.csv


In [15]:
### CAD High Severity : CADHS
for file in cadhs_train_files:
    print(file)
    data=pd.read_csv(file)
    data.drop('SYNTAX.SCORE',axis=1,inplace=True)
    train_set=data
    y_train=train_set['SYNTAX.HIGH.SEVERITY']
    X_train=train_set.iloc[:, :-1]
    ratio_under=min(y_train.value_counts())/max(y_train.value_counts())
    categorical_inds=categorical_indices(X_train)
    [X_train_rus,y_train_rus]=RUS(X_train,y_train,ratio_under)

    ### RUS
    new_filename = file.replace('_CADHS_orig', '_CADHS_rus')
    X_train_rus['SYNTAX.HIGH.SEVERITY']=y_train_rus
    X_train_rus.to_csv(new_filename,index=False)

    ### ENN
    [X_train_enn,y_train_enn]=ENN(X_train,y_train)
    new_filename = file.replace('_CADHS_orig', '_CADHS_enn')
    X_train_enn['SYNTAX.HIGH.SEVERITY']=y_train_enn
    X_train_enn.to_csv(new_filename,index=False)

    ### ROS
    [X_train_ros,y_train_ros]=ROS(X_train,y_train,1)
    new_filename = file.replace('_CADHS_orig', '_CADHS_ros')
    X_train_ros['SYNTAX.HIGH.SEVERITY']=y_train_ros
    X_train_ros.to_csv(new_filename,index=False)

    ### SMOTE
    if(len(categorical_inds)>0):
        [X_train_smote,y_train_smote]=smote_fnc(X_train,y_train,ratio=1,type_smote='categorical',categorical_col_inds=categorical_inds)
    else:
        [X_train_smote,y_train_smote]=smote_fnc(X_train,y_train,ratio=1,type_smote='numerical',categorical_col_inds=[])
    new_filename = file.replace('_CADHS_orig', '_CADHS_smote')
    X_train_smote['SYNTAX.HIGH.SEVERITY']=y_train_smote
    X_train_smote.to_csv(new_filename,index=False)
    ### SMOTE-ENN
    if(len(categorical_inds)>0):
        [X_train_smote_enn,y_train_smote_enn]=SMOTE_ENN(X_train,y_train,ratio=1,type_smote='categorical',categorical_feats=categorical_inds)
    else:
        [X_train_smote_enn,y_train_smote_enn]=SMOTE_ENN(X_train,y_train,ratio=1,type_smote='numerical',categorical_feats=[])
    new_filename = file.replace('_CADHS_orig', '_CADHS_smote_enn')
    X_train_smote_enn['SYNTAX.HIGH.SEVERITY']=y_train_smote_enn
    X_train_smote_enn.to_csv(new_filename,index=False)
    ###### Need to address correctly the indexes of the categorical columns.... 

Train_set_CADHS_orig_1.csv
Train_set_CADHS_orig_10.csv
Train_set_CADHS_orig_2.csv
Train_set_CADHS_orig_3.csv
Train_set_CADHS_orig_4.csv
Train_set_CADHS_orig_5.csv
Train_set_CADHS_orig_6.csv
Train_set_CADHS_orig_7.csv
Train_set_CADHS_orig_8.csv
Train_set_CADHS_orig_9.csv


For the Repeats

In [ ]:
os.chdir(r'.\Repeats\Rep10\Train_test_sets')
train_sets=os.listdir(os.getcwd())

#train_sets
cads_files = [f for f in train_sets if "_CADS_" in f]
cadp_files = [f for f in train_sets if "_CADP_" in f]
cadhs_files = [f for f in train_sets if "_CADHS_" in f]
cads_train_files=[f for f in cads_files if "Train_" in f]
cadp_train_files=[f for f in cadp_files if "Train_" in f]
cadhs_train_files=[f for f in cadhs_files if "Train_" in f] 


cad_all_files=[f for f in train_sets if "Whole_" in f]

### CAD Presence : CADP
for file in cadp_train_files:
    print(file)
    data=pd.read_csv(file)
    data.drop('SYNTAX.SCORE',axis=1,inplace=True)
    train_set=data
    y_train=train_set['SYNTAX.BINARY']
    X_train=train_set.iloc[:, :-1]
    ratio_under=min(y_train.value_counts())/max(y_train.value_counts())
    categorical_inds=categorical_indices(X_train)
    [X_train_rus,y_train_rus]=RUS(X_train,y_train,ratio_under)

    ### RUS
    new_filename = file.replace('_CADP_orig', '_CADP_rus')
    X_train_rus['SYNTAX.BINARY']=y_train_rus
    X_train_rus.to_csv(new_filename,index=False)

    ### ENN
    [X_train_enn,y_train_enn]=ENN(X_train,y_train)
    new_filename = file.replace('_CADP_orig', '_CADP_enn')
    X_train_enn['SYNTAX.BINARY']=y_train_enn
    X_train_enn.to_csv(new_filename,index=False)

    ### ROS
    [X_train_ros,y_train_ros]=ROS(X_train,y_train,1)
    new_filename = file.replace('_CADP_orig', '_CADP_ros')
    X_train_ros['SYNTAX.BINARY']=y_train_ros
    X_train_ros.to_csv(new_filename,index=False)

    ### SMOTE
    [X_train_smote,y_train_smote]=smote_fnc(X_train,y_train,ratio=1,type_smote='categorical',categorical_col_inds=categorical_inds)
    new_filename = file.replace('_CADP_orig', '_CADP_smote')
    X_train_smote['SYNTAX.BINARY']=y_train_smote
    X_train_smote.to_csv(new_filename,index=False)
    ### SMOTE-ENN
    [X_train_smote_enn,y_train_smote_enn]=SMOTE_ENN(X_train,y_train,ratio=1,type_smote='categorical',categorical_feats=categorical_inds)
    new_filename = file.replace('_CADP_orig', '_CADP_smote_enn')
    X_train_smote_enn['SYNTAX.BINARY']=y_train_smote_enn
    X_train_smote_enn.to_csv(new_filename,index=False)
    ###### Need to address correctly the indexes of the categorical columns.... 




### CAD Severity : CADS
for file in cads_train_files:
    print(file)
    data=pd.read_csv(file)
    data.drop('SYNTAX.SCORE',axis=1,inplace=True)
    train_set=data
    y_train=train_set['SYNTAX.SEVERITY']
    X_train=train_set.iloc[:, :-1]
    ratio_under=min(y_train.value_counts())/max(y_train.value_counts())
    categorical_inds=categorical_indices(X_train)
    [X_train_rus,y_train_rus]=RUS(X_train,y_train,ratio_under)

    ### RUS
    new_filename = file.replace('_CADS_orig', '_CADS_rus')
    X_train_rus['SYNTAX.SEVERITY']=y_train_rus
    X_train_rus.to_csv(new_filename,index=False)

    ### ENN
    [X_train_enn,y_train_enn]=ENN(X_train,y_train)
    new_filename = file.replace('_CADS_orig', '_CADS_enn')
    X_train_enn['SYNTAX.SEVERITY']=y_train_enn
    X_train_enn.to_csv(new_filename,index=False)

    ### ROS
    [X_train_ros,y_train_ros]=ROS(X_train,y_train,1)
    new_filename = file.replace('_CADS_orig', '_CADS_ros')
    X_train_ros['SYNTAX.SEVERITY']=y_train_ros
    X_train_ros.to_csv(new_filename,index=False)

    ### SMOTE
    if(len(categorical_inds)>0):
        [X_train_smote,y_train_smote]=smote_fnc(X_train,y_train,ratio=1,type_smote='categorical',categorical_col_inds=categorical_inds)
    else:
        [X_train_smote,y_train_smote]=smote_fnc(X_train,y_train,ratio=1,type_smote='numerical',categorical_col_inds=[])
    new_filename = file.replace('_CADS_orig', '_CADS_smote')
    X_train_smote['SYNTAX.SEVERITY']=y_train_smote
    X_train_smote.to_csv(new_filename,index=False)
    ### SMOTE-ENN
    if(len(categorical_inds)>0):
        [X_train_smote_enn,y_train_smote_enn]=SMOTE_ENN(X_train,y_train,ratio=1,type_smote='categorical',categorical_feats=categorical_inds)
    else:
        [X_train_smote_enn,y_train_smote_enn]=SMOTE_ENN(X_train,y_train,ratio=1,type_smote='numerical',categorical_feats=[])
    new_filename = file.replace('_CADS_orig', '_CADS_smote_enn')
    X_train_smote_enn['SYNTAX.SEVERITY']=y_train_smote_enn
    X_train_smote_enn.to_csv(new_filename,index=False)
    ###### Need to address correctly the indexes of the categorical columns.... 







### CAD High Severity : CADHS
for file in cadhs_train_files:
    print(file)
    data=pd.read_csv(file)
    data.drop('SYNTAX.SCORE',axis=1,inplace=True)
    train_set=data
    y_train=train_set['SYNTAX.HIGH.SEVERITY']
    X_train=train_set.iloc[:, :-1]
    ratio_under=min(y_train.value_counts())/max(y_train.value_counts())
    categorical_inds=categorical_indices(X_train)
    [X_train_rus,y_train_rus]=RUS(X_train,y_train,ratio_under)

    ### RUS
    new_filename = file.replace('_CADHS_orig', '_CADHS_rus')
    X_train_rus['SYNTAX.HIGH.SEVERITY']=y_train_rus
    X_train_rus.to_csv(new_filename,index=False)

    ### ENN
    [X_train_enn,y_train_enn]=ENN(X_train,y_train)
    new_filename = file.replace('_CADHS_orig', '_CADHS_enn')
    X_train_enn['SYNTAX.HIGH.SEVERITY']=y_train_enn
    X_train_enn.to_csv(new_filename,index=False)

    ### ROS
    [X_train_ros,y_train_ros]=ROS(X_train,y_train,1)
    new_filename = file.replace('_CADHS_orig', '_CADHS_ros')
    X_train_ros['SYNTAX.HIGH.SEVERITY']=y_train_ros
    X_train_ros.to_csv(new_filename,index=False)

    ### SMOTE
    if(len(categorical_inds)>0):
        [X_train_smote,y_train_smote]=smote_fnc(X_train,y_train,ratio=1,type_smote='categorical',categorical_col_inds=categorical_inds)
    else:
        [X_train_smote,y_train_smote]=smote_fnc(X_train,y_train,ratio=1,type_smote='numerical',categorical_col_inds=[])
    new_filename = file.replace('_CADHS_orig', '_CADHS_smote')
    X_train_smote['SYNTAX.HIGH.SEVERITY']=y_train_smote
    X_train_smote.to_csv(new_filename,index=False)
    ### SMOTE-ENN
    if(len(categorical_inds)>0):
        [X_train_smote_enn,y_train_smote_enn]=SMOTE_ENN(X_train,y_train,ratio=1,type_smote='categorical',categorical_feats=categorical_inds)
    else:
        [X_train_smote_enn,y_train_smote_enn]=SMOTE_ENN(X_train,y_train,ratio=1,type_smote='numerical',categorical_feats=[])
    new_filename = file.replace('_CADHS_orig', '_CADHS_smote_enn')
    X_train_smote_enn['SYNTAX.HIGH.SEVERITY']=y_train_smote_enn
    X_train_smote_enn.to_csv(new_filename,index=False)
    ###### Need to address correctly the indexes of the categorical columns.... 

Train_set_CADP_orig_1.csv
Train_set_CADP_orig_10.csv
Train_set_CADP_orig_2.csv
Train_set_CADP_orig_3.csv
Train_set_CADP_orig_4.csv
Train_set_CADP_orig_5.csv
Train_set_CADP_orig_6.csv
Train_set_CADP_orig_7.csv
Train_set_CADP_orig_8.csv
Train_set_CADP_orig_9.csv
Train_set_CADS_orig_1.csv
Train_set_CADS_orig_10.csv
Train_set_CADS_orig_2.csv
Train_set_CADS_orig_3.csv
Train_set_CADS_orig_4.csv
Train_set_CADS_orig_5.csv
Train_set_CADS_orig_6.csv
Train_set_CADS_orig_7.csv
Train_set_CADS_orig_8.csv
Train_set_CADS_orig_9.csv
Train_set_CADHS_orig_1.csv
Train_set_CADHS_orig_10.csv
Train_set_CADHS_orig_2.csv
Train_set_CADHS_orig_3.csv
Train_set_CADHS_orig_4.csv
Train_set_CADHS_orig_5.csv
Train_set_CADHS_orig_6.csv
Train_set_CADHS_orig_7.csv
Train_set_CADHS_orig_8.csv
Train_set_CADHS_orig_9.csv
